# Image Classification

A deterministic local fixture keeps transforms, loss, and reporting inspectable.

In [ ]:
from pathlib import Path
import importlib.util
lesson_rel = Path('phases/04-computer-vision/04-image-classification')
candidates = []
for start in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([start / lesson_rel / 'code/main.py', start / 'code/main.py'])
code_path = next(path.resolve() for path in candidates if path.is_file())
spec = importlib.util.spec_from_file_location('cv04_l04', code_path)
classifier = importlib.util.module_from_spec(spec)
spec.loader.exec_module(classifier)
print(code_path)

In [ ]:
import numpy as np
images, labels = classifier.synthetic_cifar(8, 3, 12, seed=4)
features = classifier.image_features(images)
weights, bias, history = classifier.train_linear_classifier(features, labels, 3, epochs=30, lr=0.6, seed=8)
predictions = np.argmax(features @ weights.T + bias, axis=1)
print(images.shape, features.shape, history[0], history[-1], np.mean(predictions == labels))

In [ ]:
mixed_x, mixed_y = classifier.mixup_batch(images[:2], labels[:2], 3, alpha=0.5, rng=np.random.default_rng(3))
print('mixup', mixed_x.shape, mixed_y.round(3).tolist())
print('report', classifier.per_class_report(classifier.confusion_matrix(labels, predictions, 3)))
try:
    classifier.cross_entropy(np.zeros((2, 3)), np.array([0, 4]))
except ValueError as error:
    print('invalid label:', error)